# Test de la stratégie B : RAG

## Objectif
Évaluer la stratégie B, qui combine un modèle de langage (LLM) avec un mécanisme de récupération d’informations (Retrieval-Augmented Generation – RAG), afin de répondre aux questions d’une FAQ administrative.  
Cette stratégie vise à améliorer la précision et la fiabilité des réponses en s’appuyant sur une base documentaire locale, contrairement à la stratégie A reposant uniquement sur le LLM.

Les tests sont réalisés à l’aide des modèles **Qwen/Qwen2.5-7B-Instruct** et **mistralai/Mistral-7B-Instruct-v0.2**.

## Critères de vérification
- **Exactitude** des informations fournies par rapport aux documents de référence.
- **Réduction des hallucinations**, notamment sur les procédures locales.
- **Pertinence** des réponses vis-à-vis des questions posées.
- **Latence** du système lors de la génération des réponses.
- **Complexité** du système 

In [6]:
from pathlib import Path

# chargement des FAQ
root_path= Path().resolve().parents[0]
faq_path = root_path/"data"/"FAQ_Base.json"
print(root_path)
print(faq_path)

D:\ProjectFolderDevAI_2025-2026\Projet_FAQ_Intelligent\Assistant_FAQ_Intelligent
D:\ProjectFolderDevAI_2025-2026\Projet_FAQ_Intelligent\Assistant_FAQ_Intelligent\data\FAQ_Base.json


In [9]:
import json
with open(faq_path, "r", encoding="utf-8") as f:
    faq_data = json.load(f)

records = faq_data["faq"] if isinstance(faq_data, dict) and "faq" in faq_data else faq_data
print(records[:2])  # Affiche les deux premiers enregistrements pour vérification

[{'id': 'EC001', 'category': 'etat_civil', 'question': 'Comment obtenir un acte de naissance ?', 'answer': "Pour obtenir un acte de naissance, vous pouvez faire la demande en ligne sur le site service-public.fr, par courrier à la mairie du lieu de naissance, ou directement au guichet. La demande est gratuite. Munissez-vous d'une pièce d'identité. Le délai de délivrance est de 3 à 10 jours ouvrés selon le mode de demande.", 'keywords': ['naissance', 'acte', 'extrait', 'certificat', 'état civil']}, {'id': 'EC002', 'category': 'etat_civil', 'question': 'Quels documents faut-il pour se marier ?', 'answer': "Pour se marier, chaque futur époux doit fournir : une pièce d'identité en cours de validité, un justificatif de domicile de moins de 3 mois, un acte de naissance de moins de 3 mois (6 mois si délivré à l'étranger), la liste des témoins (2 à 4 personnes majeures). Le dossier doit être déposé à la mairie au moins 1 mois avant la date souhaitée.", 'keywords': ['mariage', 'documents', 'doss

In [10]:
# Chargement du fichier FAQ_Base.json dans un dataframe pandas pour visialiser les données
import pandas as pd
faq_df = pd.json_normalize(records)
faq_df.head(20)

,id,category,question,answer,keywords
0,EC001,etat_civil,Comment obtenir un acte de naissance ?,"Pour obtenir un acte de naissance, vous pouvez...","[naissance, acte, extrait, certificat, état ci..."
1,EC002,etat_civil,Quels documents faut-il pour se marier ?,"Pour se marier, chaque futur époux doit fourni...","[mariage, documents, dossier, épouser, union]"
2,EC003,etat_civil,Comment déclarer un décès ?,La déclaration de décès doit être faite dans l...,"[décès, mort, déclaration, acte de décès]"
3,EC004,etat_civil,Comment conclure un PACS ?,"Pour conclure un PACS, les partenaires doivent...","[PACS, pacte civil, union, partenariat, concub..."
4,EC005,etat_civil,Comment obtenir un livret de famille ?,Le livret de famille est délivré automatiqueme...,"[livret, famille, duplicata, perte]"
5,EC006,etat_civil,Comment faire reconnaître un enfant ?,La reconnaissance d'un enfant peut être faite ...,"[reconnaissance, enfant, filiation, père, pate..."
6,EC007,etat_civil,Comment changer de prénom ?,"Depuis 2017, le changement de prénom se fait e...","[prénom, changement, modification, identité]"
7,URB001,urbanisme,Comment déposer un permis de construire ?,Le permis de construire se dépose à la mairie ...,"[permis, construire, construction, travaux, bâ..."
8,URB002,urbanisme,Quelle autorisation pour une extension de mais...,L'autorisation dépend de la surface créée : mo...,"[extension, agrandissement, surface, déclarati..."
9,URB003,urbanisme,Faut-il une autorisation pour installer une pi...,Pour une piscine hors-sol installée moins de 3...,"[piscine, bassin, autorisation, déclaration]"


In [22]:
from sentence_transformers import SentenceTransformer, util
from huggingface_hub import InferenceClient
from dotenv import load_dotenv
import os


load_dotenv()
# Initialisation du client avec le token d'authentification
token_benchmark_faq = os.getenv("token_benchmark_faq")

# Récupération des textes des FAQ
faq_texts =[f"question: {item.get('question', ' ')}\n"
            f"answer :{item.get('answer', ' ')}\n"
            f"keywords: {', '.join(item.get('keywords', []))}" for item in records]

#print(faq_texts[:2])  # Affiche le premier texte pour vérification
# Déclaration de la liste FAQ
faq_list = list(records)
#print(faq_list[0]["answer"])  # Affiche le premier élément de la liste pour vérification

# 1. Recherche sémantique
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')
faq_embeddings = embedding_model.encode(faq_texts, convert_to_tensor=True)

question = "Comment obtenir un acte de naissance?"
q_emb = embedding_model.encode(question, convert_to_tensor=True)
similarities = util.cos_sim(q_emb, faq_embeddings)[0]
top_indices = similarities.argsort(descending=True)[:3].tolist()
print("Indices des FAQ les plus similaires :", top_indices) # Affiche les indices pour vérification

# 2. Construction du contexte
context = "\n\n---\n\n".join(
                                [f"Q: {faq_list[i]['question']}\nA: {faq_list[i]['answer']}" for i in top_indices]
)
print("Contexte sélectionné :")
print(context)

# 3. Génération avec LLM
client = InferenceClient(
                         token=token_benchmark_faq,
                       
                        )
messages = [
    {"role":"system","content":
                            "Commence toujours par « Bonjour, »."
                            "Tu réponds UNIQUEMENT à partir du CONTEXTE. "
                            "Si le CONTEXTE ne contient pas l'information, réponds : "
                            "« Bonjour, je n'ai pas l'information dans la FAQ fournie. Veuillez préciser votre demande. » "
                            "Ne fais aucune supposition."},
    {"role": "user", "content": f"Contexte:\n\n{context}\n\nQuestion: {question}"}
]
response = client.chat_completion(
    model="mistralai/Mistral-7B-Instruct-v0.2",
    messages=messages,
    temperature=0.1,
    max_tokens=300
)
print("\n Réponse générée par le modèle :")
print(response.choices[0].message.content)

Indices des FAQ les plus similaires : [0, 54, 5]
Contexte sélectionné :
Q: Comment obtenir un acte de naissance ?
A: Pour obtenir un acte de naissance, vous pouvez faire la demande en ligne sur le site service-public.fr, par courrier à la mairie du lieu de naissance, ou directement au guichet. La demande est gratuite. Munissez-vous d'une pièce d'identité. Le délai de délivrance est de 3 à 10 jours ouvrés selon le mode de demande.

---

Q: Comment ouvrir un compteur d'eau ?
A: Pour ouvrir un compteur d'eau lors d'un emménagement, contactez le service des eaux de la communauté de communes au 02 XX XX XX XX ou remplissez le formulaire en ligne. Fournissez : nom de l'ancien occupant si connu, relevé du compteur à votre arrivée, RIB. La mise en service est facturée 30€.

---

Q: Comment faire reconnaître un enfant ?
A: La reconnaissance d'un enfant peut être faite avant ou après la naissance, à la mairie de votre choix. Le père non marié doit effectuer cette démarche pour établir la filiati

In [23]:
from sentence_transformers import SentenceTransformer, util
from huggingface_hub import InferenceClient
from dotenv import load_dotenv
import os


load_dotenv()
# Initialisation du client avec le token d'authentification
token_benchmark_faq = os.getenv("token_benchmark_faq")

# Récupération des textes des FAQ
faq_texts =[f"question: {item.get('question', ' ')}\n"
            f"answer :{item.get('answer', ' ')}\n"
            f"keywords: {', '.join(item.get('keywords', []))}" for item in records]

#print(faq_texts[:2])  # Affiche le premier texte pour vérification
# Déclaration de la liste FAQ
faq_list = list(records)
#print(faq_list[0]["answer"])  # Affiche le premier élément de la liste pour vérification

# 1. Recherche sémantique
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')
faq_embeddings = embedding_model.encode(faq_texts, convert_to_tensor=True)

question = "Comment obtenir un acte de naissance?"
q_emb = embedding_model.encode(question, convert_to_tensor=True)
similarities = util.cos_sim(q_emb, faq_embeddings)[0]
top_indices = similarities.argsort(descending=True)[:3].tolist()
print("Indices des FAQ les plus similaires :", top_indices) # Affiche les indices pour vérification

# 2. Construction du contexte
context = "\n\n---\n\n".join(
                                [f"Q: {faq_list[i]['question']}\nA: {faq_list[i]['answer']}" for i in top_indices]
)
print("Contexte sélectionné :")
print(context)

# 3. Génération avec LLM
client = InferenceClient(
                         token=token_benchmark_faq,
                       
                        )
messages = [
    {"role":"system","content":
                            "Commence toujours par « Bonjour, »."
                            "Tu réponds UNIQUEMENT à partir du CONTEXTE. "
                            "Si le CONTEXTE ne contient pas l'information, réponds : "
                            "« Bonjour, je n'ai pas l'information dans la FAQ fournie. Veuillez préciser votre demande. » "
                            "Ne fais aucune supposition."},
    {"role": "user", "content": f"Contexte:\n\n{context}\n\nQuestion: {question}"}
]
response = client.chat_completion(
    model="Qwen/Qwen2.5-7B-Instruct",
    messages=messages,
    temperature=0.1,
    max_tokens=300
)
print("\n Réponse générée par le modèle :")
print(response.choices[0].message.content)

Indices des FAQ les plus similaires : [0, 54, 5]
Contexte sélectionné :
Q: Comment obtenir un acte de naissance ?
A: Pour obtenir un acte de naissance, vous pouvez faire la demande en ligne sur le site service-public.fr, par courrier à la mairie du lieu de naissance, ou directement au guichet. La demande est gratuite. Munissez-vous d'une pièce d'identité. Le délai de délivrance est de 3 à 10 jours ouvrés selon le mode de demande.

---

Q: Comment ouvrir un compteur d'eau ?
A: Pour ouvrir un compteur d'eau lors d'un emménagement, contactez le service des eaux de la communauté de communes au 02 XX XX XX XX ou remplissez le formulaire en ligne. Fournissez : nom de l'ancien occupant si connu, relevé du compteur à votre arrivée, RIB. La mise en service est facturée 30€.

---

Q: Comment faire reconnaître un enfant ?
A: La reconnaissance d'un enfant peut être faite avant ou après la naissance, à la mairie de votre choix. Le père non marié doit effectuer cette démarche pour établir la filiati

In [ ]:
from sentence_transformers import SentenceTransformer, util
from huggingface_hub import InferenceClient
from dotenv import load_dotenv
import os


load_dotenv()
# Initialisation du client avec le token d'authentification
token_benchmark_faq = os.getenv("token_benchmark_faq")

# Récupération des textes des FAQ
faq_texts =[f"question: {item.get('question', ' ')}\n"
            f"answer :{item.get('answer', ' ')}\n"
            f"keywords: {', '.join(item.get('keywords', []))}" for item in records]

#print(faq_texts[:2])  # Affiche le premier texte pour vérification
# Déclaration de la liste FAQ
faq_list = list(records)
#print(faq_list[0]["answer"])  # Affiche le premier élément de la liste pour vérification

# Prompte système
SYSTEM_PROMPT = (
    "Commence toujours par « Bonjour, ».\n"
    "Utilise un ton affirmatif et direct.\n"
    "N'utilise JAMAIS de termes incertains ou modaux tels que : "
    "« peut-être », « devriez », « probablement », « il est possible que ».\n"
    "Réponds uniquement à la question posée.\n"
    "Si plusieurs éléments du contexte sont présents, utilise uniquement ceux "
    "qui répondent directement à la question.\n"
    "Tu réponds UNIQUEMENT à partir du CONTEXTE fourni.\n"
    "Si le CONTEXTE ne contient pas l'information, réponds EXACTEMENT :\n"
    "« Bonjour, je n'ai pas l'information dans la FAQ fournie. Veuillez préciser votre demande. »\n"
    "Ne fais aucune supposition."
)
# 1. Recherche sémantique
embedding_model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')
faq_embeddings = embedding_model.encode(faq_texts, convert_to_tensor=True)

question = "Comment obtenir un acte de naissance?"
q_emb = embedding_model.encode(question, convert_to_tensor=True)
similarities = util.cos_sim(q_emb, faq_embeddings)[0]
top_indices = similarities.argsort(descending=True)[:3].tolist()
print("Indices des FAQ les plus similaires :", top_indices) # Affiche les indices pour vérification

# 2. Construction du contexte
context = "\n\n---\n\n".join(
                                [f"Q: {faq_list[i]['question']}\nA: {faq_list[i]['answer']}" for i in top_indices]
)
print("Contexte sélectionné :")
print(context)

# 3. Génération avec LLM
client = InferenceClient(
                         token=token_benchmark_faq,
                       
                        )
messages = [
    {"role":"system","content":SYSTEM_PROMPT},
    {"role": "user", "content": f"Contexte:\n\n{context}\n\nQuestion: {question}"}
]
response = client.chat_completion(
    model="Qwen/Qwen2.5-7B-Instruct",
    messages=messages,
    temperature=0.1,
    max_tokens=300
)
print("\nRéponse générée par le modèle :")
print(response.choices[0].message.content)

Indices des FAQ les plus similaires : [0, 5, 4]
Contexte sélectionné :
Q: Comment obtenir un acte de naissance ?
A: Pour obtenir un acte de naissance, vous pouvez faire la demande en ligne sur le site service-public.fr, par courrier à la mairie du lieu de naissance, ou directement au guichet. La demande est gratuite. Munissez-vous d'une pièce d'identité. Le délai de délivrance est de 3 à 10 jours ouvrés selon le mode de demande.

---

Q: Comment faire reconnaître un enfant ?
A: La reconnaissance d'un enfant peut être faite avant ou après la naissance, à la mairie de votre choix. Le père non marié doit effectuer cette démarche pour établir la filiation. Documents : pièce d'identité, justificatif de domicile, acte de naissance de l'enfant (si reconnaissance après naissance). La reconnaissance est gratuite et immédiate.

---

Q: Comment obtenir un livret de famille ?
A: Le livret de famille est délivré automatiquement lors du mariage ou à la naissance du premier enfant pour les couples no